# selfattention发生在encoder部分


In [ ]:
!pip install bertviz

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 157.5/157.5 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.0/140.0 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.3/15.3 MB 93.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 95.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.1/90.1 kB 7.0 MB/s eta 0:00:00


In [ ]:
import torch
from torch import nn
import math
from bertviz.transformers_neuron_view import BertModel,BertConfig
from transformers import BertTokenizer

In [ ]:
max_length = 256
model_name = 'bert-base-uncased'
# output_attentions=True: 告诉模型在每次前向传播（Forward Pass）计算完成后，除了输出最终结果，还要返回每一层计算产生的注意力权重（Attention Weights）矩阵。
# output_hidden_states=True: 告诉模型返回内部所有 12 层 Transformer 层的隐藏状态（Hidden States），而不只是最后一层的输出。
# return_dict=True: 确保模型的输出格式是一个类似字典的 ModelOutput 对象，而不是一个基础的元组（Tuple）。这让你可以在后续代码中通过类似
config = BertConfig.from_pretrained(model_name,output_attentions=True,output_hidden_states=True,return_dict = True)
tokenizer = BertTokenizer.from_pretrained(model_name)
config.max_position_embeddings = max_length

model = BertModel(config).from_pretrained(model_name)
model = model.eval()

100%|██████████| 433/433 [00:00<00:00, 931350.58B/s]


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

100%|██████████| 440473133/440473133 [00:09<00:00, 48525384.15B/s]


In [ ]:
model.config

{
  "architectures": [
    "BertForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "finetuning_task": null,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "layer_norm_eps": 1e-12,
  "max_position_embeddings": 512,
  "model_type": "bert",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "num_labels": 2,
  "output_attentions": true,
  "output_hidden_states": false,
  "pad_token_id": 0,
  "torchscript": false,
  "type_vocab_size": 2,
  "vocab_size": 30522
}

单个头的维度 = 总维度 ÷ 头的总数

In [ ]:
att_head_size = int(model.config.hidden_size/model.config.num_attention_heads)

# 因此自定义注意力头的时候，数量大小必须是768的约数

In [ ]:
att_head_size

64

In [ ]:
model.encoder

BertEncoder(
  (layer): ModuleList(
    (0-11): 12 x BertLayer(
      (attention): BertAttention(
        (self): BertSelfAttention(
          (query): Linear(in_features=768, out_features=768, bias=True)
          (key): Linear(in_features=768, out_features=768, bias=True)
          (value): Linear(in_features=768, out_features=768, bias=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (output): BertSelfOutput(
          (dense): Linear(in_features=768, out_features=768, bias=True)
          (LayerNorm): BertLayerNorm()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
      (intermediate): BertIntermediate(
        (dense): Linear(in_features=768, out_features=3072, bias=True)
      )
      (output): BertOutput(
        (dense): Linear(in_features=3072, out_features=768, bias=True)
        (LayerNorm): BertLayerNorm()
        (dropout): Dropout(p=0.1, inplace=False)
      )
    )
  )
)

In [ ]:
model.encoder.layer[0]

BertLayer(
  (attention): BertAttention(
    (self): BertSelfAttention(
      (query): Linear(in_features=768, out_features=768, bias=True)
      (key): Linear(in_features=768, out_features=768, bias=True)
      (value): Linear(in_features=768, out_features=768, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (output): BertSelfOutput(
      (dense): Linear(in_features=768, out_features=768, bias=True)
      (LayerNorm): BertLayerNorm()
      (dropout): Dropout(p=0.1, inplace=False)
    )
  )
  (intermediate): BertIntermediate(
    (dense): Linear(in_features=768, out_features=3072, bias=True)
  )
  (output): BertOutput(
    (dense): Linear(in_features=3072, out_features=768, bias=True)
    (LayerNorm): BertLayerNorm()
    (dropout): Dropout(p=0.1, inplace=False)
  )
)

In [ ]:
model.encoder.layer[0].attention


BertAttention(
  (self): BertSelfAttention(
    (query): Linear(in_features=768, out_features=768, bias=True)
    (key): Linear(in_features=768, out_features=768, bias=True)
    (value): Linear(in_features=768, out_features=768, bias=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (output): BertSelfOutput(
    (dense): Linear(in_features=768, out_features=768, bias=True)
    (LayerNorm): BertLayerNorm()
    (dropout): Dropout(p=0.1, inplace=False)
  )
)

In [ ]:
model.encoder.layer[0].attention.self


BertSelfAttention(
  (query): Linear(in_features=768, out_features=768, bias=True)
  (key): Linear(in_features=768, out_features=768, bias=True)
  (value): Linear(in_features=768, out_features=768, bias=True)
  (dropout): Dropout(p=0.1, inplace=False)
)

# 那么如何体现多头呢？在bert中是使用分块矩阵的方法

: 定位到 BERT 模型的第 1 层 Transformer 编码器（索引从 0 开始）。定位到这一层里面的“自注意力（Self-Attention）”模块。获取这个线性层的权重矩阵参数（$W_Q$）。.T (转置): 在 PyTorch 中，nn.Linear 存储权重的默认形状是 [out_features, in_features]。使用 .T 将其转置为 [in_features, out_features]

在代码实现中，为了提高 GPU 的并行计算效率，Hugging Face 的 Transformers 库并没有为 12 个注意力头分别创建 12 个小的权重矩阵，而是将它们拼接成了一个完整的大矩阵（即 768 维）。

因为我们之前算过，每个头的维度（att_head_size）是 64，所以：

第 1 个头负责大矩阵的第 0 到 63 列。

第 2 个头负责大矩阵的第 64 到 127 列。

以此类推……

In [ ]:
model.encoder.layer[0].attention.self.query.weight.T[:,:64]

tensor([[-0.0164, -0.0326,  0.0105,  ..., -0.0186, -0.0095,  0.0112],
        [ 0.0261,  0.0346,  0.0334,  ...,  0.0482, -0.0285, -0.0349],
        [-0.0263, -0.0423,  0.0109,  ..., -0.0724, -0.0453, -0.0304],
        ...,
        [ 0.0154, -0.0527, -0.0279,  ..., -0.0434,  0.0170,  0.0217],
        [ 0.0768,  0.1393,  0.0258,  ...,  0.0385,  0.0357, -0.0631],
        [ 0.0548,  0.0078, -0.0468,  ...,  0.0423, -0.0408,  0.0212]],
       grad_fn=<SliceBackward0>)

In [ ]:
model.encoder.layer[0].attention.self.query.weight.T[:,64:128]

tensor([[-0.0112, -0.0324, -0.0615,  ..., -0.0383,  0.0031,  0.0059],
        [ 0.0260, -0.0067, -0.0616,  ...,  0.1097,  0.0029, -0.0540],
        [-0.0169,  0.0232,  0.0068,  ...,  0.0124, -0.0168,  0.0301],
        ...,
        [ 0.1083,  0.0056,  0.0968,  ...,  0.0188, -0.0171,  0.0141],
        [-0.0436, -0.1032, -0.1035,  ...,  0.0138, -0.0488, -0.0453],
        [-0.0611,  0.0224, -0.0320,  ...,  0.0376,  0.0186, -0.0482]],
       grad_fn=<SliceBackward0>)

In [ ]:
model.encoder.layer[0].attention.self.query.weight.T[:,128:192]

tensor([[ 0.0452, -0.0334, -0.0087,  ...,  0.0702,  0.0229, -0.0483],
        [ 0.0876,  0.0561,  0.1416,  ...,  0.0227, -0.0727, -0.0380],
        [ 0.0002,  0.0350, -0.0158,  ..., -0.1045,  0.0051, -0.0124],
        ...,
        [-0.0171,  0.0633, -0.0253,  ...,  0.0825,  0.0125,  0.0158],
        [-0.0526, -0.0006, -0.0323,  ...,  0.0151,  0.0097, -0.0092],
        [-0.0278,  0.0596, -0.0043,  ...,  0.0160,  0.0251, -0.0346]],
       grad_fn=<SliceBackward0>)

# 数据准备


In [ ]:
from sklearn.datasets import fetch_20newsgroups
newsgroups_train = fetch_20newsgroups(subset='train')
input_tests = tokenizer(newsgroups_train['data'][:1],truncation=True,padding=True,max_length=max_length,return_tensors='pt')

In [ ]:
input_tests['input_ids'].shape

torch.Size([1, 201])

In [ ]:
input_tests.keys()

KeysView({'input_ids': tensor([[  101,  2013,  1024,  3393,  2099,  2595,  3367,  1030, 11333,  2213,
          1012,  8529,  2094,  1012,  3968,  2226,  1006,  2073,  1005,  1055,
          2026,  2518,  1007,  3395,  1024,  2054,  2482,  2003,  2023,   999,
          1029,  1050,  3372,  2361,  1011, 14739,  1011,  3677,  1024, 10958,
          2278,  2509,  1012, 11333,  2213,  1012,  8529,  2094,  1012,  3968,
          2226,  3029,  1024,  2118,  1997,  5374,  1010,  2267,  2380,  3210,
          1024,  2321,  1045,  2001,  6603,  2065,  3087,  2041,  2045,  2071,
          4372,  7138,  2368,  2033,  2006,  2023,  2482,  1045,  2387,  1996,
          2060,  2154,  1012,  2009,  2001,  1037,  1016,  1011,  2341,  2998,
          2482,  1010,  2246,  2000,  2022,  2013,  1996,  2397, 20341,  1013,
          2220, 17549,  1012,  2009,  2001,  2170,  1037,  5318,  4115,  1012,
          1996,  4303,  2020,  2428,  2235,  1012,  1999,  2804,  1010,  1996,
          2392, 21519,  2001,

In [ ]:
## 双星号字典解包，自动拿到token_type_ids和attention_mask和对应的值
model_output = model(**input_tests)

In [ ]:
## 上一章中的三个层，last_hidden_state,pooler ,hidden_States
len(model_output)

3

In [ ]:
## bert-base-uncased的12层结构
len(model_output[-1])

12

In [ ]:
## 12层中的第一层，最接近输入
model_output[-1][0].keys()

dict_keys(['attn', 'queries', 'keys'])

In [ ]:
## 形状上来看，batch=1，12个注意力头数，201的toeknizer之后的embeddings长度
model_output[-1][0]['attn'].shape

torch.Size([1, 12, 201, 201])

In [ ]:
## dim=-1 的意思是沿着最后一个维度（即按“列”的方向，也就是对每一“行”）进行求和。
## 第一个 0：取出第 0 个 Batch（因为你只有一句话，所以取出这句话）。第二个 0：取出第 0 个注意力头（剥离出第一个 Head 的视角）。第一个 :：选中目标序列的所有词（行）。第二个 :：选中源序列的所有词（列）。
model_output[-1][0]['attn'][0,0,:,:]

tensor([[0.0053, 0.0109, 0.0052,  ..., 0.0039, 0.0036, 0.0144],
        [0.0086, 0.0041, 0.0125,  ..., 0.0045, 0.0041, 0.0071],
        [0.0051, 0.0043, 0.0046,  ..., 0.0043, 0.0045, 0.0031],
        ...,
        [0.0010, 0.0023, 0.0055,  ..., 0.0012, 0.0018, 0.0011],
        [0.0010, 0.0023, 0.0057,  ..., 0.0012, 0.0017, 0.0007],
        [0.0022, 0.0056, 0.0063,  ..., 0.0045, 0.0048, 0.0015]],
       grad_fn=<SelectBackward0>)

# from scratch

In [ ]:
emb_output = model.embeddings(input_tests['input_ids'],input_tests['token_type_ids'])

通常，emb_output 的完整形状是 [batch_size, sequence_length, hidden_size]。

在这里，输入的是单独的一段文本（比如上文中的一句话），所以 batch_size 为 1。那么 emb_output 的形状就是 [1, 201, 768]。

1: 批次大小。

201: 文本的序列长度（这段文本被分词器切分成了 201 个 Token）。

768: BERT 基础模型的默认隐藏层维度。

取 emb_output[0] 就是为了去掉第一个大小为 1 的批次维度，提取出真正的特征矩阵。降维后的形状变成了 [201, 768]，这样更方便在后续进行标准的二维矩阵乘法操作。

In [ ]:
emb_output[0].shape

torch.Size([201, 768])

In [ ]:
## 201*64
Q_first_head_first_layer = emb_output[0] @ model.encoder.layer[0].attention.self.query.weight.T[:,:att_head_size] \
              + model.encoder.layer[0].attention.self.query.bias[:att_head_size]

In [ ]:
## 201*64
K_first_head_first_layer = emb_output[0] @ model.encoder.layer[0].attention.self.key.weight.T[:,:att_head_size] \
              + model.encoder.layer[0].attention.self.key.bias[:att_head_size]

In [ ]:
## (201*64) * (64*201) ==> 201 * 201
attn_scores = torch.nn.Softmax(dim=-1)(Q_first_head_first_layer @ K_first_head_first_layer.T / math.sqrt(att_head_size))

In [ ]:
attn_scores

tensor([[0.0053, 0.0109, 0.0052,  ..., 0.0039, 0.0036, 0.0144],
        [0.0086, 0.0041, 0.0125,  ..., 0.0045, 0.0041, 0.0071],
        [0.0051, 0.0043, 0.0046,  ..., 0.0043, 0.0045, 0.0031],
        ...,
        [0.0010, 0.0023, 0.0055,  ..., 0.0012, 0.0018, 0.0011],
        [0.0010, 0.0023, 0.0057,  ..., 0.0012, 0.0017, 0.0007],
        [0.0022, 0.0056, 0.0063,  ..., 0.0045, 0.0048, 0.0015]],
       grad_fn=<SoftmaxBackward0>)

In [ ]:
V_first_head_first_layer = emb_output[0] @ model.encoder.layer[0].attention.self.value.weight.T[:,:att_head_size] \
              + model.encoder.layer[0].attention.self.value.bias[:att_head_size]

In [ ]:
attn_emb = attn_scores @ V_first_head_first_layer

In [ ]:
attn_emb.shape

torch.Size([201, 64])